### 这个notebook只针对 - CustomerComplaint进行数据预处理

### 连接数据库

In [ ]:
import re,emoji
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


### 读取数据

In [ ]:
# 读取来自CustomerComplaint的数据
customer_complaint_sql = """
SELECT * FROM "CustomerComplaint"
"""

df_cc = pd.read_sql(customer_complaint_sql, engine)

# 查看数据
df_cc

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,None
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31
...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,None


In [3]:
df_cc.columns.tolist()

['complaint_id',
 'customer_id',
 'product_id',
 'order_id',
 'complaint_type',
 'complaint_text',
 'complaint_date',
 'complaint_severity',
 'resolution_status',
 'resolution_date']

### 文本预处理 - 文本去重

In [ ]:
# ---------- 2) 时间统一、去重
def preprocess_basic_reviews(df):
    # 拼接 title + text，去重，时间解析
    df = df.copy()
    df['complaint_text'] = df.get('complaint_text', '').fillna('')
    df['text'] = df['complaint_text']
    if 'complaint_date' in df.columns:
        df['complaint_date'] = pd.to_datetime(df['complaint_date'], errors='coerce')
    if 'resolution_date' in df.columns:
        df['resolution_date'] = pd.to_datetime(df['resolution_date'], errors='coerce')
    # 去重基于 complaint_id 或 text+customer_id+date
    if 'complaint_id' in df.columns:
        df = df.drop_duplicates(subset=['complaint_id'], keep='first')
    else:
        df = df.drop_duplicates(subset=['customer_id', 'product_id', 'text'], keep='first')
    # 可选：过滤掉完全为空的文本
    df = df[df['text'].str.strip() != ''].reset_index(drop=True)

    return df

# ---------- 3) 简单文本去噪函数
def clean_text(s, 
               lower=True, 
               remove_urls=True, 
               remove_html=True, 
               remove_nonprint=True, 
               remove_emoji=True,
               keep_only_english_digits=False):

    if pd.isna(s): return ""

    text = str(s).strip()
    
    if remove_html:
        text = re.sub(r'<[^>]+>', ' ', text)
    if remove_urls:
        text = re.sub(r'http\S+|www\.\S+', ' ', text)
    
    if remove_emoji and emoji is not None:
        text = emoji.demojize(text, delimiters=(" ", " "))
        text = text.replace("_", " ")
        text = re.sub(r'[:]+', ' ', text)
    if remove_nonprint:
        text = re.sub(r'[\r\n\t]+', ' ', text)
    # 移除常见占位文本 N/A, n.a., NA, na, 等（不区分大小写）
    text = re.sub(r'\b(n/?a|n\.a\.|na)\b', ' ', text, flags=re.I)
    
    text = re.sub(r'\s+', ' ', text).strip()

    if lower:
        text = text.lower()
    if keep_only_english_digits:
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text


### 检测非英文文本内容

### 主程序

In [ ]:
# 文本预处理
customer_complaints = preprocess_basic_reviews(df_cc)
# 选择1：清洗文本列，保留原文
customer_complaints['text_clean'] = customer_complaints['text'].apply(clean_text)

# 只检测唯一文本，避免重复计算
unique_texts = customer_complaints["text_clean"].dropna().drop_duplicates()

lang_map = {}
for txt in unique_texts:
    if not isinstance(txt, str):
        lang_map[txt] = "unknown"
        continue
    txt = txt.strip()
    if len(txt) < 2:
        lang_map[txt] = "unknown"
        continue
    try:
        lang_map[txt] = detect(txt)
    except Exception:
        lang_map[txt] = "unknown"

# 回填到原表
customer_complaints["lang"] = customer_complaints["text_clean"].map(lang_map)

# 暂时不需要
# ------------ ##
# 只保留非英语
non_english_complaints = customer_complaints[
    customer_complaints["lang"].isin(["fr", "es", "de", "it", "pt", "nl", "ja", "ko", "zh-cn", "zh-tw"])
].copy()

# 增加空白翻译列，供后续人工或自动翻译填充
non_english_complaints["text_clean_translated"] = ""

# 导出
non_english_complaints.to_csv("customer_complaints_non_english.csv", index=False, encoding="utf-8-sig")

non_english_complaints.head()


,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,text_clean_translated
10,11,82829,178,266338,product,Buttons don’t press,2024-11-30,High,Escalated,2024-12-24,Buttons don’t press,buttons don’t press,fr,
84,85,43101,207,327729,product,Doesn’t recognize my fingerprint,2024-08-31,High,Resolved,2024-09-15,Doesn’t recognize my fingerprint,doesn’t recognize my fingerprint,it,
99,100,62812,144,44227,product,Case doesn’t fit,2025-01-11,Low,Resolved,2025-01-17,Case doesn’t fit,case doesn’t fit,es,
148,149,63593,161,175308,delivery,"No photo proof, no package, no delivery",2024-10-21,High,In Progress,NaT,"No photo proof, no package, no delivery","no photo proof, no package, no delivery",it,
168,169,64877,132,194554,product,Item smells burnt,2025-06-10,Low,Resolved,2025-06-16,Item smells burnt,item smells burnt,de,


In [6]:
customer_complaints.to_csv("customer_complaints_all_languages.csv", index=False, encoding="utf-8-sig")
customer_complaints

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en


In [8]:
resolution_status_dist = customer_complaints["resolution_status"].value_counts().sort_index()
print("\nresolution_status 分布：")
print(resolution_status_dist)
print("\n占比：%")
print((customer_complaints["resolution_status"].value_counts(normalize=True).sort_index() * 100).round(2))


resolution_status 分布：
resolution_status
Escalated       694
In Progress    1210
Resolved       5087
Unresolved     1009
Name: count, dtype: int64

占比：%
resolution_status
Escalated       8.67
In Progress    15.12
Resolved       63.59
Unresolved     12.61
Name: proportion, dtype: float64


### TF-IDF + Logistic Regression 训练（二分法）

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

# 1) 先准备一个词典（情感词权重）
positive_words = {
    "good": 1.0, "great": 1.5, "excellent": 2.0, "love": 2.0,
    "amazing": 2.0, "fast": 1.0, "clear": 0.8, "satisfied": 1.5,
    "quality": 0.8, "nice": 1.0, "happy": 1.2
}

negative_words = {
    "bad": -1.0, "poor": -1.5, "terrible": -2.0, "hate": -2.0,
    "slow": -1.2, "broken": -2.0, "damaged": -2.0, "issue": -1.5,
    "problem": -1.5, "delay": -1.2, "refund": -1.0, "frustrated": -1.8
}

# 2) TF-IDF
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9
)

X_tfidf = tfidf.fit_transform(customer_complaints["text_clean"].fillna(""))

feature_names = np.array(tfidf.get_feature_names_out())

# 3) 计算每条文本的情感分数
scores = []
for i in range(X_tfidf.shape[0]):
    row = X_tfidf.getrow(i)
    row_indices = row.indices
    row_data = row.data

    pos_score = 0.0
    neg_score = 0.0

    for idx, weight in zip(row_indices, row_data):
        token = feature_names[idx]
        token = token.lower()

        if token in positive_words:
            pos_score += weight * positive_words[token]
        if token in negative_words:
            neg_score += abs(weight) * abs(negative_words[token])

    # 总分：越大越正向
    sentiment_score = pos_score + neg_score
    scores.append(sentiment_score)

customer_complaints["tfidf_sentiment_score"] = scores

def label_from_score(x, pos_thresh=0.2, neg_thresh=-0.2):
    if x > pos_thresh:
        return 1
    elif x < neg_thresh:
        return 0
    else:
        return -1

customer_complaints_tfidf_score = customer_complaints.copy()

customer_complaints_tfidf_score["tfidf_sentiment_label"] = customer_complaints_tfidf_score["tfidf_sentiment_score"].apply(label_from_score)

customer_complaints_tfidf_score

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,tfidf_sentiment_score,tfidf_sentiment_label
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en,0.00000,-1
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en,0.32603,1
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en,0.00000,-1
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en,0.00000,-1
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en,0.00000,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en,0.00000,-1
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so,0.00000,-1
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en,0.00000,-1
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en,0.00000,-1


### TF-IDF + Logistic Regression提取投诉负面关键词v3 - 最佳版本

In [14]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ============================================================
# 1) 停用词和泛词过滤
# ============================================================
ENGLISH_STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "else", "for", "with",
    "from", "into", "onto", "in", "on", "at", "to", "of", "by", "as", "is",
    "are", "was", "were", "be", "been", "being", "this", "that", "these",
    "those", "it", "its", "he", "she", "they", "them", "we", "you", "i", "me",
    "my", "your", "our", "us", "do", "does", "did", "not", "no", "yes", "can",
    "could", "would", "should", "have", "has", "had", "will", "just", "too",
    "very", "more", "most", "some", "any", "all", "each", "about", "after",
    "before", "because", "under", "over", "again", "there", "here", "when",
    "where", "who", "which", "what", "how", "why", "than", "also", "only",
    "because", "through", "during", "while", "other", "same", "such", "than"
}

# 领域泛词：投诉场景中通常不适合作为“关键词”
GENERIC_WORDS = {
    "customer", "service", "product", "products", "complaint", "complaints",
    "issue", "issues", "problem", "problems", "order", "orders", "quality",
    "item", "items", "store", "seller", "time", "day", "days", "work",
    "worked", "working", "company", "experience", "buy", "bought", "want",
    "got", "received", "return", "refund", "shipping", "delivery", "package"
}

def is_valid_keyword(word: str) -> bool:
    if pd.isna(word):
        return False
    w = str(word).strip().lower()
    if len(w) <= 2:
        return False
    if not re.fullmatch(r"[a-z]+", w):
        return False
    if w in ENGLISH_STOPWORDS or w in GENERIC_WORDS:
        return False
    return True

def clean_keyword_df(df: pd.DataFrame, top_n: int = 50) -> pd.DataFrame:
    """
    过滤停用词、泛词、过短词，并按系数排序。
    这里假设 df 列：word, coef
    """
    out = df.copy()
    out["word"] = out["word"].astype(str).str.lower()
    out = out[out["word"].apply(is_valid_keyword)].copy()
    out = out.sort_values("coef", ascending=False).head(top_n)
    return out.reset_index(drop=True)

# ============================================================
# 2) 负面词典（可来自网上词典 + 业务词典）
# ============================================================
neg_keywords = {
    "bad", "poor", "terrible", "hate", "slow", "broken", "damaged", "issue",
    "problem", "delay", "refund", "frustrated", "wrong", "late", "defect",
    "unhappy", "failure", "missing", "leak", "crack", "scratch", "blur",
    "distorted", "foggy", "pain", "hurt", "tight", "loose", "fall", "drop",
    "cracked", "misaligned", "uncomfortable", "faulty", "broken", "defective",
    "incorrect", "narrow", "glare", "squeak", "scratchy", "stuck", "delay",
    "expired", "worn", "fake", "cheap", "poorly", "badly"
}

def build_negative_label(text):
    """
    二分类标签：
    1 = 负面投诉
    0 = 非负面投诉
    """
    if pd.isna(text):
        return 0
    txt = str(text).lower()
    return 1 if any(k in txt for k in neg_keywords) else 0

# ============================================================
# 3) 先构造标签
# ============================================================
# 建议先保留原始清洗文本
customer_complaints["label_negative"] = customer_complaints["text_clean"].apply(build_negative_label)

# 只保留有效文本（非空）
customer_complaints_model = customer_complaints[
    customer_complaints["text_clean"].notna() &
    (customer_complaints["text_clean"].str.len() > 0)
].copy()

print("标签分布：")
print(customer_complaints_model["label_negative"].value_counts().sort_index())

# 如果只有一个类别，无法训练模型，需要先检查
if customer_complaints_model["label_negative"].nunique() < 2:
    raise ValueError("标签只有一个类别，无法训练二分类模型，请先检查负面词典或标签构造逻辑。")

# ============================================================
# 4) 划分训练集与测试集
# ============================================================
X = customer_complaints_model["text_clean"].fillna("").astype(str)
y = customer_complaints_model["label_negative"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ============================================================
# 5) TF-IDF + Logistic Regression
# ============================================================
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    strip_accents="unicode",
    lowercase=True,
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]{2,}\b"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="liblinear"
)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("\n=== 模型评估 ===")
print(classification_report(y_test, y_pred, target_names=["non_negative", "negative"]))

# ============================================================
# 6) 提取“负面关键词”
# ============================================================
feature_names = np.array(tfidf.get_feature_names_out())
coef = model.coef_[0]  # 二分类时，coef[0] 对应 class=1 的系数；且标签 1 = negative

keywords_df = pd.DataFrame({
    "word": feature_names,
    "coef": coef
})

# 因为 label=1 表示 negative，所以系数越大，越偏向 negative
negative_keywords_df = clean_keyword_df(keywords_df, top_n=50)
negative_keywords_df = negative_keywords_df.sort_values("coef", ascending=False).reset_index(drop=True)

print("\n=== 最强负面关键词（Top 50） ===")
print(negative_keywords_df.head(20))

# 也可以直接输出整表
negative_keywords_df

# ============================================================
# 7) 可选：把模型预测结果保存回原表
# ============================================================
customer_complaints_model["tfidf_logreg_negative_pred"] = model.predict(tfidf.transform(X))
customer_complaints_model["tfidf_logreg_negative_prob"] = model.predict_proba(tfidf.transform(X))[:, 1]

# 也可以保留一个简单规则分数，便于对照
customer_complaints_model["tfidf_rule_negative_score"] = customer_complaints_model["text_clean"].apply(
    lambda t: 1 if any(k in str(t).lower() for k in neg_keywords) else 0
)

# 保存输出
customer_complaints_model.to_csv(
    "6-8-customer_complaints_model_with_predictions-by-tfidf-logreg.csv",
    index=False,
    encoding="utf-8-sig"
)

negative_keywords_df.to_csv(
    "6-8-negative_keywords_top50-by-tfidf-logreg.csv",
    index=False,
    encoding="utf-8-sig"
)

print("已保存：6-8-customer_complaints_model_with_predictions-by-tfidf-logreg.csv")
print("已保存：6-8-negative_keywords_top50-by-tfidf-logreg.csv")

customer_complaints_model


标签分布：
label_negative
0    6677
1    1323
Name: count, dtype: int64

=== 模型评估 ===
              precision    recall  f1-score   support

non_negative       1.00      1.00      1.00      1335
    negative       1.00      0.98      0.99       265

    accuracy                           1.00      1600
   macro avg       1.00      0.99      0.99      1600
weighted avg       1.00      1.00      1.00      1600


=== 最强负面关键词（Top 50） ===
             word       coef
0           wrong  12.236086
1         missing   9.384626
2            late   7.463999
3          broken   6.474739
4         delayed   6.168479
5           stuck   5.520609
6           paint   5.158533
7           delay   4.479503
8         cracked   4.386230
9           loose   4.245996
10        leaking   4.177513
11      incorrect   3.723748
12          cheap   3.708864
13  uncomfortable   3.618831
14        dropped   3.231427
15        damaged   2.986731
16        falling   2.981838
17     misaligned   2.974748
18       inflate

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,tfidf_sentiment_score,label_negative,tfidf_logreg_negative_pred,tfidf_logreg_negative_prob,tfidf_rule_negative_score
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en,0.00000,0,0,0.196761,0
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en,0.32603,1,1,0.818365,1
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en,0.00000,0,0,0.186347,0
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en,0.00000,0,0,0.142283,0
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en,0.00000,0,0,0.102577,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en,0.00000,0,0,0.181766,0
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so,0.00000,0,0,0.080262,0
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en,0.00000,0,0,0.113099,0
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en,0.00000,0,0,0.073355,0


In [15]:
negative_keywords_df

,word,coef
0,wrong,12.236086
1,missing,9.384626
2,late,7.463999
3,broken,6.474739
4,delayed,6.168479
5,stuck,5.520609
6,paint,5.158533
7,delay,4.479503
8,cracked,4.386230
9,loose,4.245996


### 跑 VADER 做无监督对比（二分法）

In [10]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    scores = analyzer.polarity_scores(str(text))
    return pd.Series({
        'vader_neg': scores['neg'],
        'vader_neu': scores['neu'],
        'vader_pos': scores['pos'],
        'vader_compound': scores['compound']  # 最常用的综合分数
    })

vader_scores = customer_complaints['text_clean'].apply(get_vader_scores)
customer_complaints_vader_score = pd.concat([customer_complaints, vader_scores], axis=1)
customer_complaints_vader_score


,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,tfidf_sentiment_score,vader_neg,vader_neu,vader_pos,vader_compound
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en,0.00000,0.000,1.000,0.0,0.0000
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en,0.32603,0.485,0.515,0.0,-0.5719
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en,0.00000,0.000,1.000,0.0,0.0000
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en,0.00000,0.000,1.000,0.0,0.0000
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en,0.00000,0.000,1.000,0.0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en,0.00000,0.286,0.714,0.0,-0.2500
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so,0.00000,0.459,0.541,0.0,-0.1779
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en,0.00000,0.239,0.761,0.0,-0.2960
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en,0.00000,0.000,1.000,0.0,0.0000


### Textblob (NaiveBayesAnalyzer)

In [11]:
from textblob import TextBlob
from textblob.sentiments import NaiveBayesAnalyzer
import pandas as pd

nb_analyzer = NaiveBayesAnalyzer()

def get_textblob_nb_scores(text):
    try:
        blob = TextBlob(str(text), analyzer=nb_analyzer)
        cls = blob.sentiment.classification   # 'pos' or 'neg'
        p_pos = blob.sentiment.p_pos          # 概率
        p_neg = blob.sentiment.p_neg
        return pd.Series({
            'tb_nb_class': cls,
            'tb_nb_p_pos': p_pos,
            'tb_nb_p_neg': p_neg
        })
    except Exception:
        return pd.Series({'tb_nb_class': None, 'tb_nb_p_pos': None, 'tb_nb_p_neg': None})

tb_nb_scores = customer_complaints['text_clean'].fillna('').astype(str).apply(get_textblob_nb_scores)
customer_complaints_tb_nb = pd.concat([customer_complaints, tb_nb_scores], axis=1)

# 可选把分类映射为 1/0
customer_complaints_tb_nb['tb_nb_label'] = customer_complaints_tb_nb['tb_nb_class'].map({'pos':1, 'neg':0})
customer_complaints_tb_nb

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,tfidf_sentiment_score,tb_nb_class,tb_nb_p_pos,tb_nb_p_neg,tb_nb_label
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en,0.00000,pos,0.863636,0.136364,1
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en,0.32603,pos,0.903890,0.096110,1
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en,0.00000,pos,0.525221,0.474779,1
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en,0.00000,pos,0.669339,0.330661,1
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en,0.00000,neg,0.476462,0.523538,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en,0.00000,pos,0.881487,0.118513,1
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so,0.00000,neg,0.457945,0.542055,0
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en,0.00000,neg,0.401395,0.598605,0
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en,0.00000,pos,0.545356,0.454644,1


### Textblob (默认的PatternAnalyzer)

In [12]:
from textblob import TextBlob
from textblob.sentiments import PatternAnalyzer
import pandas as pd

pattern_analyzer = PatternAnalyzer()

def get_textblob_scores(text):
    try:
        blob = TextBlob(str(text), analyzer=pattern_analyzer)
        return pd.Series({
            'tb_pa_polarity': blob.sentiment.polarity,      # 连续值，范围约在 [-1, 1]
            'tb_pa_subjectivity': blob.sentiment.subjectivity
        })
    except Exception:
        return pd.Series({'tb_pa_polarity': None, 'tb_pa_subjectivity': None})

tb_scores = customer_complaints['text_clean'].fillna('').astype(str).apply(get_textblob_scores)
customer_complaints_tb_pa = pd.concat([customer_complaints, tb_scores], axis=1)

def polarity_to_label(p, pos_thresh=0.05, neg_thresh=-0.05):
    if p is None:
        return -1
    if p > pos_thresh:
        return 1
    if p < neg_thresh:
        return 0
    return -1

customer_complaints_tb_pa['tb_pa_label'] = customer_complaints_tb_pa['tb_pa_polarity'].apply(polarity_to_label)
customer_complaints_tb_pa

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,tfidf_sentiment_score,tb_pa_polarity,tb_pa_subjectivity,tb_pa_label
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en,0.00000,0.0,0.0,-1
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en,0.32603,0.2,0.4,1
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en,0.00000,0.1,0.4,1
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en,0.00000,0.0,0.0,-1
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en,0.00000,0.0,0.0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en,0.00000,0.0,0.0,-1
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so,0.00000,0.0,0.0,-1
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en,0.00000,0.0,0.0,-1
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en,0.00000,0.0,0.0,-1


### 模型保存

In [16]:
import sys

print(sys.executable)
print(sys.version)

d:\miniconda_envs\junliangvenv_nlp1\python.exe
3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]


In [17]:
import importlib.util

spec = importlib.util.find_spec("torch")

print(spec.origin)

d:\miniconda_envs\junliangvenv_nlp1\Lib\site-packages\torch\__init__.py


### 测试GPU的连接性

In [18]:
from transformers import pipeline
import torch
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import classification_report, f1_score

device = 0 if torch.cuda.is_available() else -1
print("使用设备：", "GPU" if device == 0 else "CPU")

d:\miniconda_envs\junliangvenv_nlp1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


使用设备： GPU


In [19]:
import torch, sys
print("python:", sys.executable)
print("torch:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda)
print("torch.backends.cudnn.version():", torch.backends.cudnn.version())

python: d:\miniconda_envs\junliangvenv_nlp1\python.exe
torch: 2.5.1+cu121
torch.cuda.is_available(): True
torch.version.cuda: 12.1
torch.backends.cudnn.version(): 90100


### BERT模型 A：cardiffnlp/twitter-roberta-base-sentiment-latest（英文强）

In [20]:
from tqdm import tqdm
import pandas as pd

cardiffnlp_local_model_path = r"D:\huggingface_models\cardiffnlp-twitter-roberta-safe"

pipe_roberta = pipeline(
    "sentiment-analysis",
    model=cardiffnlp_local_model_path,
    tokenizer=cardiffnlp_local_model_path,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=8,          # 4GB 显存建议 8 或 16
)

# 准备文本
texts = customer_complaints['text_clean'].fillna('').astype(str).tolist()
batch_size = 32  # 根据显存/速度调整

rows = []
for i in tqdm(range(0, len(texts), batch_size), desc='RoBERTa predict'):
    batch = texts[i:i+batch_size]
    results = pipe_roberta(batch)  # 返回 dict 列表，通常包含 'label' 和 'score'
    for r in results:
        rows.append({
            'roberta_label': r.get('label'),
            'roberta_score': r.get('score')
        })

# 构造结果 DataFrame（保持与原表相同索引顺序）
df_roberta = pd.DataFrame(rows, index=customer_complaints.index)

# 合并回原表，生成新表名
customer_complaints_roberta = pd.concat([customer_complaints, df_roberta], axis=1)

# 可选：映射为数值标签（Negative -> 0; Neutral -> 1; Positive -> 2, 其它 -> -1）
def map_label(l):
    if l is None: return -1
    l = str(l).lower()
    if '0' in l or 'negative' in l:
        return 0
    elif '1' in l or 'neutral' in l:
        return 1
    elif '2' in l or 'positive' in l:
        return 2
    return -1

customer_complaints_roberta['roberta_label_mapped'] = customer_complaints_roberta['roberta_label'].apply(map_label)

# 查看前几行
customer_complaints_roberta

RoBERTa predict: 100%|██████████| 250/250 [00:10<00:00, 22.93it/s]


,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,tfidf_sentiment_score,label_negative,roberta_label,roberta_score,roberta_label_mapped
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en,0.00000,0,neutral,0.625848,1
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en,0.32603,1,negative,0.930640,0
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en,0.00000,0,negative,0.868095,0
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en,0.00000,0,negative,0.727496,0
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en,0.00000,0,neutral,0.638551,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en,0.00000,0,neutral,0.647541,1
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so,0.00000,0,negative,0.751289,0
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en,0.00000,0,negative,0.888174,0
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en,0.00000,0,negative,0.560011,0


### 提取负面高频词(使用IntegratedGradients+captum) 

In [20]:
import re
from collections import Counter
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import IntegratedGradients

# --------------------------
# 1) 加载模型
# --------------------------
local_model_path = r"D:\huggingface_models\cardiffnlp-twitter-roberta-safe"

tokenizer = AutoTokenizer.from_pretrained(local_model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(local_model_path, local_files_only=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

target_label = 0  # Negative

# 直接拿 embedding 层
embedding_layer = model.roberta.embeddings.word_embeddings

def forward_fn(embeds, attention_mask=None):
    outputs = model(
        inputs_embeds=embeds,
        attention_mask=attention_mask
    )
    return outputs.logits

ig = IntegratedGradients(forward_fn)

# --------------------------
# 2) 负面样本
# --------------------------
neg_df = customer_complaints_roberta[
    customer_complaints_roberta["roberta_label_mapped"] == 0
].copy()

neg_texts = neg_df["text_clean"].fillna("").astype(str).tolist()
neg_texts = neg_texts[:20]

token_score_counter = Counter()
token_count_counter = Counter()

for text in neg_texts:
    if not text.strip():
        continue

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # 把 token ids 转成 embeddings
    input_embeds = embedding_layer(input_ids)

    attributions = ig.attribute(
        inputs=input_embeds.to(device),
        baselines=torch.zeros_like(input_embeds).to(device),
        target=target_label,
        additional_forward_args=(attention_mask,),
        n_steps=10,
        return_convergence_delta=False
    )

    # 每个 token 的 attribution = embedding 维度上的绝对值求和
    token_attr = attributions[0].abs().sum(dim=-1).detach().cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    for tok, attr in zip(tokens, token_attr):
        if tok in {"<s>", "</s>", "[CLS]", "[SEP]", "[PAD]", "<pad>"}:
            continue
        if tok.startswith("##"):
            tok = tok[2:]
        tok = tok.lower()

        if len(tok) <= 2:
            continue
        if not re.fullmatch(r"[a-z]+", tok):
            continue

        attr_value = float(attr)
        if attr_value <= 0:
            continue

        token_score_counter[tok] += attr_value
        token_count_counter[tok] += 1

# --------------------------
# 3) 汇总
# --------------------------
top_keywords = pd.DataFrame([
    {"keyword": k, "ig_score": v, "count": token_count_counter.get(k, 0)}
    for k, v in token_score_counter.items()
]).sort_values(["ig_score", "count"], ascending=[False, False])

top_keywords = top_keywords.reset_index(drop=True)
print(top_keywords.head(50))

     keyword  ig_score  count
0       sort  9.705900      1
1       this  9.696638      3
2       item  8.415466      1
3      cloth  8.257365      1
4      aying  7.837200      1
5        the  6.811667      2
6       very  6.376266      1
7      doesn  5.756070      1
8   received  5.496269      1
9     graded  5.109216      1
10      just  4.360292      1
11    attery  4.276996      1
12   package  4.236225      1


In [21]:
top_keywords

,keyword,ig_score,count
0,sort,9.705900,1
1,this,9.696638,3
2,item,8.415466,1
3,cloth,8.257365,1
4,aying,7.837200,1
5,the,6.811667,2
6,very,6.376266,1
7,doesn,5.756070,1
8,received,5.496269,1
9,graded,5.109216,1


### 提取负面关键词(使用KeyBERT) - 第一种输出模式

In [ ]:
import re
from collections import Counter
import pandas as pd
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

# --------------------------
# 1) 负面样本筛选
# --------------------------
neg_df = customer_complaints_roberta[
    customer_complaints_roberta["roberta_label_mapped"] == 0
].copy()

neg_df = neg_df.dropna(subset=["text_clean"]).copy()
neg_df["text_clean"] = neg_df["text_clean"].astype(str)

# 只处理前 N 条文本
max_samples = 200   # 这里控制需要处理的负面样本数量，改成你想要的值
neg_texts = [t.strip() for t in neg_df["text_clean"].tolist() if str(t).strip()][:max_samples]
# neg_texts = neg_texts[:200]
print("neg_df count =", len(neg_df))
print("neg_texts count =", len(neg_texts))

# --------------------------
# 2) 加载 KeyBERT
# --------------------------
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

sentence_model = SentenceTransformer(
    r"D:\huggingface_models\sentence-transformers-all-MiniLM-L6-v2",
    device=device
)

kw_model = KeyBERT(model=sentence_model)

# --------------------------
# 3) 提取关键词
# --------------------------
keyword_counter = Counter()
keyword_score = Counter()

for i, text in enumerate(neg_texts):  # 先看前10条
    print(f"\n===== sample {i} =====")
    print("text length:", len(text))
    print("text:", text[:200])

    try:
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            use_maxsum=False,
            use_mmr=False, 
            top_n=5,
        )
        print("raw keywords count:", len(keywords))
        print("raw keywords:", keywords)

        if len(keywords) == 0:
            print("这条文本没有抽到关键词，原因可能是：文本太短/噪声太多/模型不适配")
            continue

        for kw, score in keywords:
            kw_clean = str(kw).strip().lower()
            print("before filter:", kw_clean, "score:", score)

            if not kw_clean:
                print("filtered: empty")
                continue

            if len(kw_clean) <= 2:
                print("filtered: too short")
                continue

            keyword_counter[kw_clean] += 1
            keyword_score[kw_clean] += float(score)
            print("added:", kw_clean)

    except Exception as e:
        print("KeyBERT error:", e)
        break

    print("current keyword_counter len:", len(keyword_counter))
    print("current keyword_counter top:", keyword_counter.most_common(10))



# --------------------------
# 4) 汇总结果
# --------------------------
if not keyword_counter:
    print("没有提取到任何关键词，可能是：")
    print("1) 负面样本为空")
    print("2) KeyBERT 提取结果都被过滤掉了")
    print("3) 文本太短或全是噪声")
else:
    top_keywords = pd.DataFrame([
        {
            "keyword": kw,
            "count": keyword_counter.get(kw, 0),
            "score": keyword_score.get(kw, 0.0)
        }
        for kw in keyword_counter
    ])

    top_keywords = top_keywords.sort_values(
        ["score", "count"],
        ascending=[False, False]
    ).reset_index(drop=True)

    print(top_keywords.head(50))
    # 保存到 CSV
    output_path = r"output\6-8-negative_keywords_summary-by-twitter-roberta-and-keybert.csv"
    top_keywords.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"\n关键词汇总已保存到：{output_path}")

neg_df count = 5427
neg_texts count = 200

===== sample 0 =====
text length: 38
text: this delay messed up my whole schedule
raw keywords count: 5
raw keywords: [('messed schedule', 0.7259), ('delay messed', 0.7079), ('delay', 0.6027), ('schedule', 0.5844), ('messed', 0.2651)]
before filter: messed schedule score: 0.7259
added: messed schedule
before filter: delay messed score: 0.7079
added: delay messed
before filter: delay score: 0.6027
added: delay
before filter: schedule score: 0.5844
added: schedule
before filter: messed score: 0.2651
added: messed
current keyword_counter len: 5
current keyword_counter top: [('messed schedule', 1), ('delay messed', 1), ('delay', 1), ('schedule', 1), ('messed', 1)]

===== sample 1 =====
text length: 25
text: item is completely warped
raw keywords count: 5
raw keywords: [('completely warped', 0.7644), ('warped', 0.6776), ('item completely', 0.4002), ('item', 0.2932), ('completely', 0.1038)]
before filter: completely warped score: 0.7644
added: compl

### 提取负面关键词(使用KeyBERT) - 第二种输出模式

In [22]:
# 注意：文本太长会影响速度和内存，先用前 2000 条试一下
all_text = " ".join(neg_texts[:2000])

print("语料级抽取：拼接文本长度 =", len(all_text))

try:
    corpus_keywords = kw_model.extract_keywords(
        all_text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        use_mmr=True,
        diversity=0.6,
        top_n=40
    )

    print("\n=== 语料级关键词 ===")
    print(corpus_keywords)

    corpus_df = pd.DataFrame(
        [
            {"keyword": kw, "score": float(score)}
            for kw, score in corpus_keywords
        ]
    )

    # 过滤太短词和空值
    corpus_df = corpus_df[
        corpus_df["keyword"].astype(str).str.strip().str.len() > 2
    ].copy()

    corpus_df["keyword"] = corpus_df["keyword"].astype(str).str.lower()

    # 保存到 CSV
    corpus_output_path = r"output\6-8-negative_keywords_corpus-level-by-keybert.csv"
    corpus_df.to_csv(corpus_output_path, index=False, encoding="utf-8-sig")
    print(f"\n语料级关键词已保存到：{corpus_output_path}")

except Exception as e:
    print("语料级抽取失败:", e)

语料级抽取：拼接文本长度 = 6573

=== 语料级关键词 ===
[('overcharged billed', 0.5292), ('charge changed', 0.4959), ('owe packaging', 0.4665), ('late fee', 0.3914), ('zipper broke', 0.3858), ('item defective', 0.3644), ('package stuck', 0.3528), ('warped cloth', 0.3364), ('messed schedule', 0.3277), ('want refund', 0.3205), ('account locked', 0.2996), ('replacement immediately', 0.2983), ('pattern battery', 0.2819), ('match billing', 0.2747), ('renewal notice', 0.2543), ('wrong tracking', 0.247), ('shipping case', 0.2462), ('headphones bent', 0.2448), ('service', 0.228), ('dealing horrible', 0.2055), ('edges received', 0.2039), ('buzzing noise', 0.2037), ('confirmation ignored', 0.1924), ('fixed driver', 0.192), ('authorize delivery', 0.1872), ('neighbor waited', 0.1867), ('transit update', 0.1847), ('filter came', 0.1595), ('inflated food', 0.1569), ('way reasonable', 0.152), ('fell sent', 0.1488), ('resolution emails', 0.1409), ('headache photo', 0.1386), ('furious order', 0.108), ('balance zero', 0.10

In [55]:
print("neg_texts 数量:", len(neg_texts))
print("keyword_counter 数量:", len(keyword_counter))

neg_texts 数量: 200
keyword_counter 数量: 624


In [54]:
top_keywords

,keyword,count,score
0,delivery,9,4.5240
1,package,9,4.0151
2,billing,6,3.7488
3,order,6,3.4265
4,charged,7,3.4116
...,...,...,...
619,supposed,1,0.1769
620,specifically,1,0.1601
621,does,1,0.1387
622,use,1,0.1375


In [23]:
import gc
gc.collect()

451

### BERT模型 B：nlptown/bert-base-multilingual-uncased-sentiment（支持英/法/西）

In [24]:
nlptown_bert_local_model_path = r"D:\huggingface_models\bert-base-multilingual-uncased-sentiment"

pipe_multi = pipeline(
    "sentiment-analysis",
    model=nlptown_bert_local_model_path,
    tokenizer=nlptown_bert_local_model_path,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=8,
)

def predict_multilingual(texts):
    results = pipe_multi(texts)
    labels = []
    for r in results:
        # 输出是 '1 star' ~ '5 star'
        star = int(r['label'].split()[0])
        if star >= 4:
            labels.append(1)      # 正面
        elif star <= 2:
            labels.append(0)      # 负面
        else:
            labels.append(-1)     # 中性（3 star）
    return labels

In [25]:
from tqdm import tqdm
import pandas as pd

nlptown_bert_local_model_path = r"D:\huggingface_models\bert-base-multilingual-uncased-sentiment"

pipe_multi = pipeline(
    "sentiment-analysis",
    model=nlptown_bert_local_model_path,
    tokenizer=nlptown_bert_local_model_path,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=8,
)

# 确保已有 pipe_multi（nlptown 模型）和 customer_complaints['text_clean']
texts = customer_complaints['text_clean'].fillna('').astype(str).tolist()
batch_size = 32  # 根据显存/速度调整

rows = []
for i in tqdm(range(0, len(texts), batch_size), desc='Multilingual predict'):
    batch = texts[i:i+batch_size]
    results = pipe_multi(batch)  # 返回每条 {'label': '1 star', 'score': float}
    for r in results:
        label_str = r.get('label', '')
        score = r.get('score', None)
        try:
            star = int(str(label_str).split()[0])
        except Exception:
            star = None
        # 映射：1-5 star -> -1/0/1 标注（你可自定义）
        if star is None:
            mapped = -1
        elif star >= 4:
            mapped = 1
        elif star <= 2:
            mapped = 0
        else:
            mapped = -1
        rows.append({
            'multi_star': star,
            'multi_label_raw': label_str,
            'multi_score': score,
            'multi_label_mapped': mapped
        })

df_multi = pd.DataFrame(rows, index=customer_complaints.index)
customer_complaints_multilingual = pd.concat([customer_complaints, df_multi], axis=1)

# 查看前几行
customer_complaints_multilingual

Multilingual predict: 100%|██████████| 250/250 [00:10<00:00, 22.97it/s]


,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,tfidf_sentiment_score,label_negative,multi_star,multi_label_raw,multi_score,multi_label_mapped
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en,0.00000,0,4,4 stars,0.351931,1
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en,0.32603,1,1,1 star,0.531991,0
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en,0.00000,0,1,1 star,0.763690,0
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en,0.00000,0,2,2 stars,0.468744,0
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en,0.00000,0,1,1 star,0.656309,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en,0.00000,0,1,1 star,0.568288,0
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so,0.00000,0,1,1 star,0.517534,0
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en,0.00000,0,1,1 star,0.469286,0
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en,0.00000,0,1,1 star,0.406008,0


In [26]:
print(customer_complaints_multilingual['multi_star'].value_counts().sort_index())

multi_star
1    5891
2    1005
3     589
4     126
5     389
Name: count, dtype: int64


In [33]:
customer_complaints_multilingual.columns

Index(['complaint_id', 'customer_id', 'product_id', 'order_id',
       'complaint_type', 'complaint_text', 'complaint_date',
       'complaint_severity', 'resolution_status', 'resolution_date', 'text',
       'text_clean', 'lang', 'tfidf_sentiment_score', 'label_negative',
       'multi_star', 'multi_label_raw', 'multi_score', 'multi_label_mapped'],
      dtype='object')

### BERT模型 C：distilbert-base-uncased-finetuned-sst-2-english（最轻量）

In [27]:
# 假设已定义好 pipe_distil 和 device（和你原来的那段一致）
import pandas as pd

distilbert_local_model_path = r"D:\huggingface_models\distilbert-base-uncased-finetuned-sst-2-english"

pipe_distil = pipeline(
    "sentiment-analysis",
    model=distilbert_local_model_path,
    tokenizer=distilbert_local_model_path,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=16,         # 这个模型更小，batch 可以大一点
)

texts = customer_complaints['text_clean'].fillna('').astype(str)
batch_size = 64   # 根据显存/速度调整

_label_map = {'NEGATIVE': 0, 'POSITIVE': 1}

labels = []
scores = []

for start in range(0, len(texts), batch_size):
    batch = texts.iloc[start:start + batch_size].tolist()
    results = pipe_distil(batch)                # 每项为 {'label':..., 'score':...}
    for r in results:
        lab = r.get('label')
        sc = r.get('score', None)
        labels.append(_label_map.get(lab, -1))  # 不认识的映射为 -1
        scores.append(sc)

# 构造与原表同索引的结果表并合并
df_distil = pd.DataFrame({'distil_label': labels, 'distil_score': scores}, index=texts.index)
customer_complaints_distil = pd.concat([customer_complaints, df_distil], axis=1)

# 查看
# customer_complaints_distil[['text_clean', 'distil_label', 'distil_score']]
customer_complaints_distil

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,text,text_clean,lang,tfidf_sentiment_score,label_negative,distil_label,distil_score
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,Fabric is see-through,fabric is see-through,en,0.00000,0,1,0.993169
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,This delay messed up my whole schedule,this delay messed up my whole schedule,en,0.32603,1,0,0.999741
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,Item is completely warped,item is completely warped,en,0.00000,0,0,0.999175
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,Cloth is fraying at the edges,cloth is fraying at the edges,en,0.00000,0,0,0.999091
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,I didn’t make this purchase,i didn’t make this purchase,en,0.00000,0,0,0.999289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,Torn fabric out of the box,torn fabric out of the box,en,0.00000,0,0,0.997449
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,Way too noisy,way too noisy,so,0.00000,0,0,0.999625
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,This bill just showed up with no explanation.,this bill just showed up with no explanation.,en,0.00000,0,0,0.999585
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,What happened to my next-day delivery?,what happened to my next-day delivery?,en,0.00000,0,0,0.999691


In [28]:
customer_complaints_distil['distil_label'].value_counts().sort_index()

distil_label
0    7343
1     657
Name: count, dtype: int64

In [29]:
import pandas as pd

# 自定义分箱（根据需要调整区间）
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ['[0.0,0.2)', '[0.2,0.4)', '[0.4,0.6)', '[0.6,0.8)', '[0.8,1.0]']

# 先处理缺失值（可选），然后分箱
customer_complaints_distil['distil_score_bin'] = pd.cut(
    customer_complaints_distil['distil_score'],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

# 统计每个区间的数量与占比
counts = customer_complaints_distil['distil_score_bin'].value_counts().sort_index()
pct = (counts / counts.sum() * 100).round(2)
pd.concat([counts, pct], axis=1, keys=['count', 'pct'])

,count,pct
distil_score_bin,,
"[0.0,0.2)",0,0.00
"[0.2,0.4)",0,0.00
"[0.4,0.6)",73,0.91
"[0.6,0.8)",157,1.96
"[0.8,1.0]",7770,97.12


### BERT模型评估

In [30]:
def evaluate_model(predict_func, X_test, y_test, model_name="Model"):
    batch_size = 32
    all_preds = []
    
    for i in tqdm(range(0, len(X_test), batch_size), desc=model_name):
        batch = X_test.iloc[i:i+batch_size].tolist()
        preds = predict_func(batch)
        all_preds.extend(preds)
    
    preds = pd.Series(all_preds, index=X_test.index)
    
    # 过滤中性（-1）
    mask = preds != -1
    print(f"\n===== {model_name} 评估结果 =====")
    print(classification_report(y_test[mask], preds[mask], 
                                target_names=['Negative', 'Positive']))
    print("Macro F1:", f1_score(y_test[mask], preds[mask], average='macro'))
    print("Negative F1:", f1_score(y_test[mask], preds[mask], pos_label=0))
    
    return preds

### 最终输出结果 v2

In [ ]:
import pandas as pd
import math

def normalize_preds(preds):
    out = []
    for p in preds:
        if p is None:
            out.append(-1)
        elif isinstance(p, float) and math.isnan(p):
            out.append(-1)
        else:
            try:
                out.append(int(p))
            except:
                out.append(-1)
    return out

# 1) 先统一整理成一个列表
model_dfs = [
    ("TFIDF_LogReg", customer_complaints_tfidf_score),
    ("VADER", customer_complaints_vader_score),
    ("TextBlob_Pattern", customer_complaints_tb_pa),
    ("TextBlob_NB", customer_complaints_tb_nb),
    ("Twitter_RoBERTa", customer_complaints_roberta),
    ("Multilingual_BERT", customer_complaints_multilingual),
    ("DistilBERT", customer_complaints_distil),
]

# # 2) 取所有 dataframe 的公共列
# common_cols = set.intersection(*(set(df.columns) for _, df in model_dfs))

# # 3) 用第一个 dataframe 的公共列作为基底
# base_df = model_dfs[0][1][list(common_cols)].copy().reset_index(drop=True)

# 按第一个 dataframe 的原列顺序保留公共列
base_cols = model_dfs[0][1].columns
common_cols = [
    c for c in base_cols
    if all(c in df.columns for _, df in model_dfs[1:])
]

base_df = model_dfs[0][1][common_cols].copy().reset_index(drop=True)

# 4) 把其他 dataframe 多出来的列拼接进去
for model_name, df in model_dfs[1:]:
    extra_cols = [c for c in df.columns if c not in common_cols]

    if not extra_cols:
        continue

    extra_df = df[extra_cols].copy().reset_index(drop=True)
    extra_df.columns = [f"{model_name}_{c}" for c in extra_cols]

    base_df = pd.concat([base_df, extra_df], axis=1)

# 5) 最终输出
final_df = base_df.copy()
final_df.to_csv(
    "output/6-8-customer_complaints_all_model_score.csv",
    index=False,
    encoding="utf-8-sig"
)
final_df

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,...,Twitter_RoBERTa_roberta_label_mapped,Multilingual_BERT_label_negative,Multilingual_BERT_multi_star,Multilingual_BERT_multi_label_raw,Multilingual_BERT_multi_score,Multilingual_BERT_multi_label_mapped,DistilBERT_label_negative,DistilBERT_distil_label,DistilBERT_distil_score,DistilBERT_distil_score_bin
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,...,1,0,4,4 stars,0.351931,1,0,1,0.993169,"[0.8,1.0]"
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,...,0,1,1,1 star,0.531991,0,1,0,0.999741,"[0.8,1.0]"
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,...,0,0,1,1 star,0.763690,0,0,0,0.999175,"[0.8,1.0]"
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,...,0,0,2,2 stars,0.468744,0,0,0,0.999091,"[0.8,1.0]"
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,...,1,0,1,1 star,0.656309,0,0,0,0.999289,"[0.8,1.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,...,1,0,1,1 star,0.568288,0,0,0,0.997449,"[0.8,1.0]"
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,...,0,0,1,1 star,0.517534,0,0,0,0.999625,"[0.8,1.0]"
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,...,0,0,1,1 star,0.469286,0,0,0,0.999585,"[0.8,1.0]"
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,...,0,0,1,1 star,0.406008,0,0,0,0.999691,"[0.8,1.0]"


In [38]:
final_df

,complaint_id,customer_id,product_id,order_id,complaint_type,complaint_text,complaint_date,complaint_severity,resolution_status,resolution_date,...,Twitter_RoBERTa_roberta_label_mapped,Multilingual_BERT_label_negative,Multilingual_BERT_multi_star,Multilingual_BERT_multi_label_raw,Multilingual_BERT_multi_score,Multilingual_BERT_multi_label_mapped,DistilBERT_label_negative,DistilBERT_distil_label,DistilBERT_distil_score,DistilBERT_distil_score_bin
0,1,89041,86,20896,product,Fabric is see-through,2024-06-17,Low,Resolved,2024-06-20,...,1,0,4,4 stars,0.351931,1,0,1,0.993169,"[0.8,1.0]"
1,2,59757,121,52323,delivery,This delay messed up my whole schedule,2025-01-29,High,Resolved,2025-02-17,...,0,1,1,1 star,0.531991,0,1,0,0.999741,"[0.8,1.0]"
2,3,106392,95,263055,product,Item is completely warped,2025-02-14,Low,Unresolved,NaT,...,0,0,1,1 star,0.763690,0,0,0,0.999175,"[0.8,1.0]"
3,4,43029,72,231019,product,Cloth is fraying at the edges,2024-04-14,Low,Resolved,2024-04-16,...,0,0,2,2 stars,0.468744,0,0,0,0.999091,"[0.8,1.0]"
4,5,197,136,394965,billing,I didn’t make this purchase,2024-12-28,Low,Resolved,2024-12-31,...,1,0,1,1 star,0.656309,0,0,0,0.999289,"[0.8,1.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,14446,114,102011,product,Torn fabric out of the box,2023-12-30,Low,Escalated,2024-01-07,...,1,0,1,1 star,0.568288,0,0,0,0.997449,"[0.8,1.0]"
7996,7997,44436,59,433865,product,Way too noisy,2025-01-05,High,Resolved,2025-02-05,...,0,0,1,1 star,0.517534,0,0,0,0.999625,"[0.8,1.0]"
7997,7998,86344,43,193433,billing,This bill just showed up with no explanation.,2024-01-16,High,Resolved,2024-02-10,...,0,0,1,1 star,0.469286,0,0,0,0.999585,"[0.8,1.0]"
7998,7999,132784,184,115802,delivery,What happened to my next-day delivery?,2025-03-15,Low,In Progress,NaT,...,0,0,1,1 star,0.406008,0,0,0,0.999691,"[0.8,1.0]"


In [40]:
import gc
gc.collect()

816